In [7]:
!pip -q install --upgrade huggingface_hub 
!apt -q install git -y
!pip -q install groq
!pip -q install python-dotenv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Reading package lists...
Building dependency tree...
Reading state information...
git is already the newest version (1:2.17.1-1ubuntu0.18).
0 upgraded, 0 newly installed, 0 to remove and 58 not upgraded.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
import os
import json


def load_level_windows(file_path, window_size=50, stride=1):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    level_rows = [line.rstrip() for line in lines[1:]]
    height = len(level_rows)
    width = len(level_rows[0])
    
    windows = []
    for start_col in range(0, width - window_size + 1, stride):
        window = []
        for row in level_rows:
            window.append(row[start_col:start_col + window_size])
        windows.append(window)
    
    return windows

def print_window(window):
    print('[')
    for line in window:
        print(f" '{line}',")
    print(']')

In [ ]:
file_path = "/home/pressprexx/Code/MarioLLM/mario-dataset/full_level/full_level.txt" 
windows_full = load_level_windows(file_path)

print(f"Total number of windows: {len(windows_full)}")
print("\nExample window:")
print_window(windows_full[0]) 

In [9]:
def process_all_levels(level_path):
    """
    Process all txt files in the txt directory and its subdirectories.
    Returns a dictionary where keys are file paths and values are lists of windows.
    """
    all_level_windows = {}
    
    # Walk through all directories under txt/
    for root, _, files in os.walk(level_path):
        for file in files:
            if file.endswith('.txt'):
                file_path = os.path.join(root, file)
                try:
                    windows = load_level_windows(file_path)
                    all_level_windows[file_path] = windows
                    # print(f"Processed {file_path}: {len(windows)} windows")
                except Exception as e:
                    print(f"Error processing {file_path}: {str(e)}")
    
    print(f"\nTotal levels processed: {len(all_level_windows)}")
    total_windows = sum(len(windows) for windows in all_level_windows.values())
    print(f"Total windows across all levels: {total_windows}")
    
    return all_level_windows

In [17]:
level_windows = process_all_levels('/home/pressprexx/Code/MarioLLM/mario-dataset/txt')


Total levels processed: 112
Total windows across all levels: 17071


In [18]:
# output_path = './level_windows.json'
# with open(output_path, 'w') as f:
#     json.dump(level_windows, f, indent=4)

In [10]:
with open('./notebooks/level_windows.json', 'r') as f:
    level_windows = json.load(f)

In [12]:
from mario_gpt.prompter import Prompter
from mario_gpt.prompt_adapter import PromptAdapter
from transformers import pipeline
from groq import Groq
from tqdm import tqdm
from transformers import AutoTokenizer
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv()

# Pre-initialize the LLM model
# login(token=os.getenv("HUGGINGFACE_TOKEN"))
# llm_model = pipeline("text-generation", model="meta-llama/Llama-3.1-8B", device=0)
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
tokenizer_path = "shyamsn97/Mario-GPT2-700-context-length"
level_tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
prompter = Prompter(level_tokenizer=level_tokenizer)
adapter = PromptAdapter(llm=client, prompter=prompter)

In [14]:
adapted_data = {}

for file_path, windows in tqdm(list(level_windows.items())[:1], desc="Processing levels"):
# for file_path, windows in tqdm(level_windows.items(), desc="Processing levels"): # Rodar em tudo

    level_data = []
    for window in tqdm(windows, desc=f"Processing windows for {file_path}", leave=False):
        try:
            evolved_prompt, base_prompt, combined_prompt = adapter.new_prompter('\n'.join(window), use_groq=True)
            window_data = {
                "window": window,
                "evolved_prompt": evolved_prompt, 
                "base_prompt": base_prompt,       
                "combined_prompt": combined_prompt
            }
            level_data.append(window_data)
        except Exception as e:
            print(f"Error processing window in {file_path}: {str(e)}")
            continue
    
    adapted_data[file_path] = level_data


output_path = './adapted_data_test.json'
with open(output_path, 'w') as f:
    json.dump(adapted_data, f, indent=4)

print(f"Saved adapted prompts to: {output_path}")

Processing levels: 100% 1/1 [00:00<00:00,  2.86it/s]

Saved adapted prompts to: ./adapted_data_test.json
